In [1]:
# 필요한 라이브러리를 가져옵니다.
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Qwen-1.8B-Chat 모델과 토크나이저를 로드합니다.
# trust_remote_code=True는 Hugging Face Hub에서 모델 코드를 직접 다운로드하는 것을 허용합니다.
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen-1_8B-Chat", trust_remote_code=True)

# "device_map='auto'"는 모델을 로드할 때 GPU가 있으면 자동으로 GPU를 사용하고, 없으면 CPU를 사용하도록 설정합니다.
# .eval()은 모델을 추론 모드로 설정하여 일관된 결과를 얻게 합니다.
print("모델을 로드하는 중입니다... 잠시만 기다려 주세요.")
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen-1_8B-Chat",
    device_map="auto",
    trust_remote_code=True
).eval()
print("모델 로드가 완료되었습니다.")

/data/2_data_server/cv-07/anaconda3/envs/qwen_1/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/data/2_data_server/cv-07/anaconda3/envs/qwen_1/lib/python3.8/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
A new version of the following files was downloaded from https://huggingface.co/Qwen/Qwen-1_8B-Chat:
- tokenization_qwen.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


모델을 로드하는 중입니다... 잠시만 기다려 주세요.


A new version of the following files was downloaded from https://huggingface.co/Qwen/Qwen-1_8B-Chat:
- configuration_qwen.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
/data/2_data_server/cv-07/anaconda3/envs/qwen_1/lib/python3.8/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
A new version of the following files was downloaded from https://huggingface.co/Qwen/Qwen-1_8B-Chat:
- qwen_generation_utils.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/Qwen/Qwen-1_8B-Chat:
- cpp_kernels.py
. Make sur

모델 로드가 완료되었습니다.


In [2]:
# 전체 파라미터 수를 계산합니다.
# model.parameters()로 모든 파라미터를 가져와 각 파라미터의 요소 수(p.numel())를 모두 더합니다.
total_params = sum(p.numel() for p in model.parameters())

# 훈련 가능한 파라미터 수만 계산합니다.
# p.requires_grad가 True인 파라미터만 골라 요소 수를 더합니다.
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\n--- 파라미터 정보 ---")
# f'{...:,}'는 숫자에 3자리마다 쉼표를 넣어줍니다.
print(f"전체 파라미터 수 (Total Parameters): {total_params:,}")
print(f"훈련 가능한 파라미터 수 (Trainable Parameters): {trainable_params:,}")
print("--------------------")


--- 파라미터 정보 ---
전체 파라미터 수 (Total Parameters): 1,836,828,672
훈련 가능한 파라미터 수 (Trainable Parameters): 1,836,828,672
--------------------


In [3]:
caption = "Apples, oranges, and other fruits are on display in a store."
question = "What types of fruits are visible in the image?"
choices = """A) Bananas and grapes placed in baskets
B) Apples and oranges displayed on the counter
C) Peaches and plums in a wooden crate
D) Pears and lemons arranged neatly"""

# ------------------------------------------------------------------
# Construct the prompt in English.
# ------------------------------------------------------------------
# The system prompt clearly defines the model's role.
system_prompt = "You are a helpful assistant that answers questions based on the given context."

# The user prompt is structured with Context, Question, and Choices.
user_prompt = f"""Read the following context and choose the most appropriate answer from the choices.

# Context
{caption}

# Question
{question}

# Choices
{choices}

# Answer: 
"""

# ------------------------------------------------------------------
# Generate the response using the model.chat function.
# ------------------------------------------------------------------
# Since this is the first turn, 'history' is set to None.
response, history = model.chat(tokenizer, user_prompt, history=None, system=system_prompt)

# ------------------------------------------------------------------
# Print the final result.
# ------------------------------------------------------------------
print("\n--- Model's Answer (Example 1) ---")
print(response)
print("------------------------------------")


--- Model's Answer (Example 1) ---
The most appropriate answer is B) Apples and oranges displayed on the counter. The context mentions that apples, oranges, and other fruits are on display in a store, so it can be inferred that these fruits are visible on the counter. The other options (bananas and grapes, peaches and plums, pears and lemons) do not make sense in the given context as they are not mentioned to be present in the store.
------------------------------------


In [6]:
# !pip install pandas

In [8]:
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import re
from tqdm import tqdm

def extract_answer_robustly(response_text: str) -> str:
    """
    모델의 다양한 형태의 서술형 답변 속에서 정답(A, B, C, D)을 추출하는 함수.
    
    Args:
        response_text: 모델이 생성한 전체 텍스트.

    Returns:
        추출된 정답 알파벳 (A, B, C, D) 또는 'N/A'.
    """
    # 응답을 대문자로 변환하여 일관성 유지
    text = response_text.upper()

    # 패턴 1: 'A)', '(A)', 'OPTION A', 'ANSWER IS A', 'ANSWER: A' 등의 명시적인 패턴을 먼저 찾습니다.
    # \b는 단어 경계를 의미하여, 'APPLE'의 'A'와 'A)'의 'A'를 구분합니다.
    patterns = {
        'A': r'\bA\b|\(A\)|\bOPTION A|\bANSWER IS A|\bANSWER: A',
        'B': r'\bB\b|\(B\)|\bOPTION B|\bANSWER IS B|\bANSWER: B',
        'C': r'\bC\b|\(C\)|\bOPTION C|\bANSWER IS C|\bANSWER: C',
        'D': r'\bD\b|\(D\)|\bOPTION D|\bANSWER IS D|\bANSWER: D'
    }

    # A, B, C, D 순서대로 패턴을 찾아 가장 먼저 나오는 것을 정답으로 간주
    for choice, pattern in patterns.items():
        if re.search(pattern, text):
            return choice
            
    # 만약 위 패턴으로도 찾지 못하면, 텍스트의 맨 앞에 나오는 알파벳을 찾습니다. (최후의 수단)
    match = re.search(r'^[A-D]', response_text.strip())
    if match:
        return match.group(0)

    return 'N/A' # 어떤 경우에도 정답을 찾지 못하면 N/A 반환



"""
test.csv와 submission.csv를 읽어, Qwen 모델로 정답을 추론하고
submission.csv의 answer 열을 채우고 model_output 열을 추가하여 다시 저장합니다.
"""
# --- 1. 파일 경로 설정 ---
test_data_path = '/data/2_data_server/cv-07/dice/2025_samsung_challenge/finetuning/unilm/beit3/output_coco_captioning_inference/test_with_caption.csv'
submission_path = '/data/2_data_server/cv-07/dice/2025_samsung_challenge/finetuning/unilm/beit3/output_coco_captioning_inference/sample_submission.csv'

# --- 2. 모델 및 토크나이저 로드 ---
print("Step 1: Qwen-1.8B-Chat 모델과 토크나이저를 로드합니다...")
try:
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen-1_8B-Chat", trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen-1_8B-Chat",
        device_map="auto",
        trust_remote_code=True
    ).eval()
except Exception as e:
    print(f"모델 로드 중 오류: {e}. 인터넷 연결 및 Hugging Face 상태를 확인하세요.")
    # return
print("모델 로드가 완료되었습니다.\n")

# --- 3. 데이터 로드 및 병합 ---
print("Step 2: 데이터 파일을 로드하고 병합합니다...")
try:
    test_df = pd.read_csv(test_data_path)
    submission_df = pd.read_csv(submission_path)
    merged_df = pd.merge(submission_df, test_df, on='ID', how='left')
except FileNotFoundError as e:
    print(f"오류: 파일을 찾을 수 없습니다 - {e}")
    # return
print(f"총 {len(merged_df)}개의 데이터를 처리합니다.\n")

# --- 4. 질의응답 및 정답 예측 ---
print("Step 3: 각 ID에 대해 정답 예측을 시작합니다...")
predictions = []  # 추출된 정답(A,B,C,D)을 저장할 리스트
raw_outputs = []  # 모델의 원본 답변을 저장할 리스트

for index, row in tqdm(merged_df.iterrows(), total=merged_df.shape[0], desc="Predicting"):
    caption = row['caption']
    question = row['Question']
    choices = f"A) {row['A']}\nB) {row['B']}\nC) {row['C']}\nD) {row['D']}"

    # ✨ 수정된 부분: 모델이 한 글자만 답하도록 프롬프트 강화
    system_prompt = "You are a highly intelligent QA model. Your task is to select the single best answer from the given choices based on the context."
    user_prompt = f"""Based on the context, answer the question by choosing the correct option.
Your response must be ONLY the capital letter of the correct choice (A, B, C, or D).
Do not provide any explanations, introductory text, or any other information.

# Context:
{caption}

# Question:
{question}

# Choices:
{choices}

# Answer:
"""
    
    try:
        response, _ = model.chat(tokenizer, user_prompt, history=None, system=system_prompt)
        
        # ✨ 수정된 부분: 모델 원본 답변 저장 및 개선된 정답 추출
        raw_outputs.append(response) # 원본 답변 저장
        answer = extract_answer_robustly(response) # 개선된 함수로 정답 추출

    except Exception as e:
        print(f"\nID {row['ID']} 처리 중 오류 발생: {e}")
        raw_outputs.append("MODEL_ERROR")
        answer = 'Error'
    
    predictions.append(answer)



Step 1: Qwen-1.8B-Chat 모델과 토크나이저를 로드합니다...


/data/2_data_server/cv-07/anaconda3/envs/qwen_1/lib/python3.8/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
The model is automatically converting to bf16 for faster inference. If you want to disable the automatic precision, please manually add bf16/fp16/fp32=True to "AutoModelForCausalLM.from_pretrained".
Try importing flash-attention for faster inference...
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.27it/s]


모델 로드가 완료되었습니다.

Step 2: 데이터 파일을 로드하고 병합합니다...
총 852개의 데이터를 처리합니다.

Step 3: 각 ID에 대해 정답 예측을 시작합니다...


Predicting: 100%|██████████| 852/852 [07:24<00:00,  1.92it/s]


In [ ]:
def extract_answer_final(response_text: str) -> str:
    """
    모델의 답변에서 가장 가능성 높은 정답(A, B, C, D)을 추출합니다.
    모호한 단어(e.g., 'a')가 아닌 명확한 패턴을 우선적으로 찾습니다.
    """
    text = response_text.strip()
    
    # 패턴 1: "(A)", "A)", "A." 등 명확한 마커가 있는 경우.
    # 가장 먼저 나타나는 패턴을 찾습니다.
    # 정규표현식: (선택)여는괄호 + (A,B,C,D 중 하나) + (선택)닫는괄호 + (필수)닫는괄호 또는 마침표
    # \b는 단어 경계를 의미하여, 단어 중간에 있는 A,B,C,D가 잡히는 것을 방지합니다.
    pattern1 = r'\b\(?([A-D])\)?[\.)]'
    matches = list(re.finditer(pattern1, text, re.IGNORECASE))
    if matches:
        # 텍스트에서 가장 먼저 등장하는 패턴을 정답으로 인정
        return matches[0].group(1).upper()
        
    # 패턴 2: "Answer: A" 와 같이 명시적인 구문이 있는 경우
    pattern2 = r'Answer\s*:\s*([A-D])'
    match = re.search(pattern2, text, re.IGNORECASE)
    if match:
        return match.group(1).upper()

    # 패턴 3: 답변 전체가 오직 A, B, C, D 중 하나의 글자로만 이루어진 경우
    pattern3 = r'^\s*([A-D])\s*$'
    match = re.match(pattern3, text, re.IGNORECASE)
    if match:
        return match.group(1).upper()
        
    # 모든 패턴에서 정답을 찾지 못한 경우
    return 'N/A'
    
# --- 5. 결과 업데이트 및 파일 저장 ---
# 예측된 정답과 모델 원본 출력을 DataFrame에 추가
submission_df['answer'] = predictions
submission_df['model_output'] = raw_outputs

# 새로운 csv 파일 만들기
csv_path_ = '/data/2_data_server/cv-07/dice/2025_samsung_challenge/finetuning/unilm/beit3/output_coco_captioning_inference/submission_with_model_output.csv'

# model_output은 제외하고 저장
submission_df[['ID', 'answer']].to_csv(csv_path_, index=False)

print("\nStep 4: 예측이 완료되었습니다. 결과를 submission.csv 파일에 저장합니다.")
try:
    submission_df.to_csv(submission_path, index=False)
    print(f"'{submission_path}' 파일이 성공적으로 업데이트되었습니다.")
    print("\n업데이트된 파일 내용 (상위 5개):")
    # ID, answer, model_output 열을 모두 보여줍니다.
    print(submission_df.head())
except Exception as e:
    print(f"결과를 파일로 저장하는 중 오류가 발생했습니다: {e}")


Step 4: 예측이 완료되었습니다. 결과를 submission.csv 파일에 저장합니다.
'/data/2_data_server/cv-07/dice/2025_samsung_challenge/finetuning/unilm/beit3/output_coco_captioning_inference/sample_submission.csv' 파일이 성공적으로 업데이트되었습니다.

업데이트된 파일 내용 (상위 5개):
         ID answer                                       model_output
0  TEST_000      B    B) Apples and oranges displayed on the counter.
1  TEST_001      A      A) Chinese, famous for dumplings and noodles.
2  TEST_002      B  B) For physical exercise and scenic views of t...
3  TEST_003      A  B) Swimwear, suggesting a tropical and beach-l...
4  TEST_004      A  The correct answer is A) Landing from a jump o...
